# 🛒 Apriori — Association Rule Mining — Quick Notes
---
> **Type:** Unsupervised | **Task:** Association Rules | **Dataset:** `market_basket`

## 📌 Key Points
- Finds **"if this → then that"** rules in transactions
- Classic example: **Market Basket Analysis** (Amazon "frequently bought together")
- Three key metrics:
  - **Support** = how often itemset appears in all transactions
  - **Confidence** = given LHS, how often RHS appears
  - **Lift** = how much better than random (Lift > 1 = useful rule)
- **Apriori principle**: all subsets of frequent itemset must also be frequent
- Slow on large item sets → use FP-Growth instead

## 📊 Dataset: `market_basket`
| Info | Value |
|---|---|
| Rows | 7501 transactions |
| Columns | Up to 20 items per row |
| Format | Each row = one customer's basket |

In [ ]:
import pandas as pd
import numpy as np

# 1. Load
df = pd.read_csv("/content/market_basket - market_basket.csv", header=None)
print("Shape:", df.shape)
df.head()

In [ ]:
# 2. Install apyori
!pip install apyori -q

In [ ]:
from apyori import apriori

# 3. Build transaction list
transactions = []
for i in range(0, 7501):
    transactions.append([str(df.values[i, j]) for j in range(0, 20)])

# 4. Run Apriori
#   Support = (3 items * 7 days) / 7501 ≈ 0.003
#   Confidence = 0.2 (20% of the time LHS → RHS)
#   Lift = 3 (3x better than random)
rules = apriori(
    transactions,
    min_support    = 0.003,
    min_confidence = 0.2,
    min_lift       = 3,
    min_length     = 2
)

# 5. Extract & display results
results = list(rules)
print(f"Total rules found: {len(results)}")

# Clean display
def inspect(results):
    lhs         = [tuple(res[2][0][0])[0] for res in results]
    rhs         = [tuple(res[2][0][1])[0] for res in results]
    support     = [res[1]                  for res in results]
    confidence  = [res[2][0][2]            for res in results]
    lift        = [res[2][0][3]            for res in results]
    return list(zip(lhs, rhs, support, confidence, lift))

rules_df = pd.DataFrame(inspect(results),
    columns=['Left Hand Side','Right Hand Side','Support','Confidence','Lift'])
rules_df = rules_df.sort_values('Lift', ascending=False).reset_index(drop=True)
print(rules_df.head(10))

## 🔧 Key Metrics
| Metric | Formula | Meaning |
|---|---|---|
| **Support** | freq(A∪B) / N | How common is the itemset |
| **Confidence** | freq(A∪B) / freq(A) | Given A, probability of B |
| **Lift** | Confidence / Support(B) | How much better than chance |

## 🥇 Remember
- Lift > 1 → positive association ✅
- Lift = 1 → independent (no association)
- Lift < 1 → negative association ❌
- `min_support` = (avg items/basket × days) / total transactions
- Sort by **Lift** to find strongest rules
- `apyori` returns generator → convert with `list(rules)`